In [0]:
%run /Workspace/Users/azuser7248_mml.local@karthikirisoutlook.onmicrosoft.com/capstone-datasets/00_config

In [0]:
import requests

scd_config_url = "https://raw.githubusercontent.com/Santhoshpec/capstone-datasets/refs/heads/main/configs/scd_config.json"

scd_config = requests.get(
    scd_config_url
).json()

scd_config

In [0]:
from pyspark.sql import Row
from datetime import datetime

def write_scd_audit(
        table_name,
        new_records,
        changed_records,
        status):

    audit_df = spark.createDataFrame([
        Row(
            table_name=table_name,
            run_timestamp=datetime.now(),
            new_records=new_records,
            changed_records=changed_records,
            status=status
        )
    ])

    (
        audit_df
        .write
        .format("delta")
        .mode("append")
        .save(
            f"{base_path}/audit/scd_audit/"
        )
    )

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


def run_scd_type2(config):

    entity = config["entity"]
    business_key = config["business_key"]
    surrogate_key = config["surrogate_key"]
    compare_columns = config["compare_columns"]

    source_path = (
        f"{base_path}/silver/{entity}/"
    )

    target_path = (
        f"{base_path}/gold/dimensions/dim_{entity}_scd/"
    )

    try:

        print(f"\nProcessing : {entity}")

        source_df = (
            spark.read
            .format("delta")
            .load(source_path)
        )

        # FIRST LOAD CHECK

        try:

            target_df = (
                spark.read
                .format("delta")
                .load(target_path)
            )

            target_df.limit(1).collect()

            first_load = False

        except:

            first_load = True

            print("First Load :", first_load)

        # FIRST LOAD

        if first_load:

            first_load_df = (
                source_df
                .withColumn(
                    surrogate_key,
                    row_number().over(
                        Window.orderBy(business_key)
                    )
                )
                .withColumn(
                    "effective_start_date",
                    current_date()
                )
                .withColumn(
                    "effective_end_date",
                    lit(None).cast("date")
                )
                .withColumn(
                    "is_current",
                    lit(True)
                )
            )

            (
                first_load_df
                .write
                .format("delta")
                .mode("overwrite")
                .save(target_path)
            )

            write_scd_audit(
                f"dim_{entity}_scd",
                first_load_df.count(),
                0,
                "INITIAL_LOAD"
            )

            print(f"Initial Load Completed : {entity}")

            return

        # CURRENT RECORDS

        target_current = (
            target_df
            .filter(
                col("is_current") == True
            )
        )

        # HASHES

        source_hash = (
            source_df
            .withColumn(
                "hash_value",
                sha2(
                    concat_ws(
                        "|",
                        *[
                            col(c).cast("string")
                            for c in compare_columns
                        ]
                    ),
                    256
                )
            )
        )

        target_hash = (
            target_current
            .withColumn(
                "hash_value",
                sha2(
                    concat_ws(
                        "|",
                        *[
                            col(c).cast("string")
                            for c in compare_columns
                        ]
                    ),
                    256
                )
            )
        )

        # NEW RECORDS

        new_records = (
            source_hash.alias("s")
            .join(
                target_hash.alias("t"),
                business_key,
                "left_anti"
            )
        )

        # CHANGED RECORDS

        changed_records = (
            source_hash.alias("s")
            .join(
                target_hash.alias("t"),
                business_key
            )
            .filter(
                col("s.hash_value")
                !=
                col("t.hash_value")
            )
        )

        new_count = new_records.count()
        changed_count = changed_records.count()

        # EXPIRE OLD RECORDS

        changed_keys = (
            changed_records
            .select(business_key)
            .distinct()
        )

        expired_df = (
            target_df.alias("t")
            .join(
                changed_keys.alias("c"),
                business_key,
                "left"
            )
            .withColumn(
                "is_current",
                when(
                    col(f"c.{business_key}").isNotNull(),
                    False
                ).otherwise(
                    col("is_current")
                )
            )
            .withColumn(
                "effective_end_date",
                when(
                    col(f"c.{business_key}").isNotNull(),
                    current_date()
                ).otherwise(
                    col("effective_end_date")
                )
            )
            .drop(
                col(f"c.{business_key}")
            )
        )

        # MAX SK

        max_sk = (
            target_df
            .agg(
                max(surrogate_key)
            )
            .collect()[0][0]
        )

        if max_sk is None:
            max_sk = 0

        # CHANGED INSERT

        changed_insert = (
            changed_records
            .select("s.*")
            .drop("hash_value")
            .withColumn(
                surrogate_key,
                row_number().over(
                    Window.orderBy(business_key)
                ) + max_sk
            )
            .withColumn(
                "effective_start_date",
                current_date()
            )
            .withColumn(
                "effective_end_date",
                lit(None).cast("date")
            )
            .withColumn(
                "is_current",
                lit(True)
            )
        )

        # NEW INSERT

        new_insert = (
            new_records
            .drop("hash_value")
            .withColumn(
                surrogate_key,
                row_number().over(
                    Window.orderBy(business_key)
                ) + max_sk + changed_count
            )
            .withColumn(
                "effective_start_date",
                current_date()
            )
            .withColumn(
                "effective_end_date",
                lit(None).cast("date")
            )
            .withColumn(
                "is_current",
                lit(True)
            )
        )

        final_df = (
            expired_df
            .unionByName(changed_insert)
            .unionByName(new_insert)
        )

        (
            final_df
            .write
            .format("delta")
            .mode("overwrite")
            .option(
                f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
                storage_account_key
            )
            .save(target_path)
        )

        write_scd_audit(
            f"dim_{entity}_scd",
            new_count,
            changed_count,
            "SUCCESS"
        )

        print(f"SCD Type 2 Completed : {entity}")
        print(f"New Records : {new_count}")
        print(f"Changed Records : {changed_count}")

    except Exception as e:

        write_scd_audit(
            f"dim_{entity}_scd",
            0,
            0,
            f"FAILED : {str(e)}"
        )

        raise e

In [0]:
for config in scd_config:

    run_scd_type2(config)

In [0]:
customer_dim_df = (
    spark.read
    .format("delta")
    .option(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    .load(
        f"{base_path}/gold/dimensions/dim_customers_scd/"
    )
)

customer_dim_df.display()

In [0]:
product_dim_df = (
    spark.read
    .format("delta")
    .option(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    .load(
        f"{base_path}/gold/dimensions/dim_products_scd/"
    )
)

product_dim_df.display()